# Logging and Hyperparameter Search using WandB

**Before we start**

- This tutorial is rendered from a Jupyter notebook that is hosted on GitHub. If you want to run the code yourself, you can find the notebook and configuration files [here](https://github.com/neuralhydrology/neuralhydrology/tree/master/examples/06-Finetuning).
- To be able to run this notebook locally, you need to download the publicly available CAMELS US rainfall-runoff dataset and a publicly available extensions for hourly forcing and streamflow data. See the [Data Prerequisites Tutorial](data-prerequisites.nblink) for a detailed description on where to download the data and how to structure your local dataset folder. Note the special [section](data-prerequisites.nblink#CAMELS-US-catchment-attributes) with additional requirements for this tutorial.
- In addition to these data pre-requisites, also an existing [WandB](https://wandb.ai) is required.

This notebook shows how WandB can be used to:
1) log the model metrics (similar to the default tensorboard logger)
2) test a bunch of hyperparameter combinations and find a good set.

WandB can be thought of as an evolution of tensorboard. In its basic form it provides the same metric logging functionality in a nice
web-based interface in addition to a web-storage to store the checkpoints or other artifacts created during a run. On top of that it offers
advanced organisational and comparison features.
The general structure in WandB is `Project/Run` where a run is just one experiment (i.e. one training run) and a project is a collection of runs.

Sweeps are WandB's solution to automatically test a bunch of hyperparameter configurations and find the best one. Out of the box it
implements three search strategies, iterating over the search space:
- "grid": tests each possible combination, i.e., instructs WandB to do an exhaustive search
- "random": randomly selects some configurations until a budget of allowed combinations is reached.
- "bayes": tries to estimate how strongly the different parameters contribute to the performance of the model and then searches in the direction of
    the most promising candidates.

Before we replace the tensorboard logging with the WandB equivalent, we need to install WandB.

# Install dependencies
Independent of how you've installed 'neuralhydrology`, just run
```bash
pip install wandb
```
in your terminal to install. `neuralhydrology` also defines a group of optional dependencies called "wandb". So if you prefer installing the WandB dependencies `neuralhydrology` specifies then either run
```bash
pip install neuralhydrology[wandb]
# or if you installed the package in editable mode
pip install -e .[wandb]
```

Note, however, that all versions are equivalent.

# Logging to WandB
The `neuralhydrology` repository supports tensorboard logging out of the box and provides WandB as an alternative (the actual file implementing the logging functionality is `neuralhydrology/training/wandb_logger.py`).
Logging is controlled through the following properties:
- `logger_type` defines whether the metrics shall be logged to `"tensorboard"` or `"wandb"`. By default it is set to `tensorboard` and if you want to disable logging altogether set it to `null`, `~` or leave it blank.
- `wandb_project` defines the project name you want to log the metrics of your runs to. If it does not exist yet inside WandB, it will be created automatically (for a more detailed dive into projects see [WandB-Projects](https://docs.wandb.ai/guides/track/project-page/)).

The relevant section in the config file thus might look like this:
```yaml
logger_type: wandb
wandb_project: my-example-project  # is created automatically if it doesn't exist yet
```

Before starting the script make sure that you are currently logged into wandb by running the following command in your terminal
```bash
wandb login
```

Now run the following two cells to start a training. After the data-loading has finished, you should see something like this in the output:
```
Tracking run with wandb version 0.21.0
Run data is saved locally in /path/to/neuralhydrology/examples/07-WandB/runs/test_run_0807_131018/wandb/wandb/run-20250708_131022-jjokhktf
Syncing run helpful-snowflake-1 to Weights & Biases (docs)
View project at https://wandb.ai/<your-username>/my-example-project
View run at https://wandb.ai/<your-username>/my-example-project/runs/jjokhktf
```

Once you click on the link shown in the line starting with "View run at ...", you'll see the training progress of the model in the metrics that also would be logged
to tensorboard.

In [ ]:
from pathlib import Path

import torch
from neuralhydrology.nh_run import start_run

In [ ]:
# by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
if torch.cuda.is_available() or torch.backends.mps.is_available():
    start_run(config_file=Path("1_basin.yml"))

# fall back to CPU-only mode
else:
    start_run(config_file=Path("1_basin.yml"), gpu=-1)


# Hyperparameter tuning
In hyperparameter tuning, you specify a list of possible values for the hyperparameters (like the `learning_rate`, or the `hidden_size`) of interest to you and create a sweep from them. WandB manages this sweep and
dispatches each of the possible configurations to the worker processes, called agents.

So first we are going to set up a small sweep configuration that finds the best setting for the `learning_rate` and the `hidden_size` out of $9$ possible combinations.
In addition to the possible value assignments to these parameters, we have to specify the hyperparameter search method, the name of
the sweep and the metric by which WandB shall measure the fitness of each configuration.

In [ ]:
sweep_config = {
    'method': 'grid',  # Can be 'grid', 'random', or 'bayes'
    'name': 'my_hyperparameter_search',  # Give your sweep a meaningful name
    'metric': {
        'name': 'valid/median_nse',  # Specify which metric shall be used as measure
        'goal': 'maximize'  # plus the direction in which it shall be pushed
    },
    'parameters': {
        'learning_rate': {
            'values': [0.001, 0.0005, 0.0001]
        },
        'hidden_size': {
            'values': [64, 128, 256]  # actually sensible
        },
    }
}

Using this sweep configuration, we call `wandb.sweep()` pass the config to it and specify the name of the project again in which
we want the sweep to be located.

In [ ]:
import wandb

wandb.login()  # Ensure that you are currently logged in

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="my-example-project")

print(f"Created sweep with ID: {sweep_id}")
print(f"To run sweep agents, use:")
print(f"wandb agent {sweep_id}")

Now, all we have to do is to start an agent, that duly waits for WandB to send it a new configuration to test. We do that by defining a function that
is called every time there is a new configuration available, retrieves it and then executes the test routine. In our case the test routine
is simply executing a training run. In addition to the test function, the agent also needs to know the sweep it is expected to
receive configurations from.
Inside the function, we then retrieve the values to test using `wandb.config`.

In [ ]:
from neuralhydrology.utils.config import Config
from neuralhydrology.nh_run import start_training

def train():
    """Training function called by wandb.agent for each sweep run."""
    # Load base config
    base_config = Config(Path("1_basin.yml"))

    # Initialize wandb run
    wandb.init(dir=base_config.run_dir)

    # Get sweep parameters from wandb
    sweep_params = dict(wandb.config)

    # Update the base configuration with the "to-be-tested" values
    base_config.update_config(sweep_params)

    # Now call the training function
    start_training(base_config)


In [ ]:
wandb.agent(sweep_id, train)

Both of these steps, creating the sweep and starting the agent, are reflected in the repository:
- `neuralhydrology/nh_wandb_sweep.py` creates a sweep and is also the file to list the "to-be-checked" values for the hyperparameters.
- `neuralhydrology/nh_run.py` started in the "sweep" mode with `--sweep-id <sweep-id>` passed as argument will start the agent, apply the received values to a base
    config and then start the training with this config. For convenience, the individual configuration files are written to the "sweep" directory next to the base
    configuration file.

# Doing it all in the terminal

That is to create a sweep first modify the configuration in `neuralhydrology/nh_wandb_sweep.py` and then run
```bash
python neuralhydrology/nh_wandb_sweep.py
```
This script should already print the full command to execute the agent at the end of its output:
```bash
python neuralhydrology/nh_run.py sweep --sweep-id=<sweep-id>
```